# 상권 데이터 기본 탐색 (EDA 이전 단계)

이 노트북의 목적은 딱 하나: **우리가 가진 CSV 3개가 각각 뭘 담고 있는지, 어떻게 생겼는지** 감을 잡는 것.

아직 분석(인사이트 도출)은 하지 않는다. 순서대로 셀을 실행하면서 출력을 눈으로 확인하면 됨.

- `seoul_commercial_district_master_2025.csv` : 상권의 '위치 정보' (어느 자치구/동에 있는 상권인지)
- `seoul_commercial_district_stores_2025.csv` : 상권별 '점포/업종/개폐업' 정보 (제일 큰 파일, 30만 행)
- `seoul_commercial_district_attraction_facilities_2025.csv` : 상권별 '주변 시설' 정보 (지하철역, 학교, 병원 등)

세 파일 모두 `상권_코드`라는 공통 열쇠(key)로 서로 연결(조인)할 수 있음.

## 0. 준비: 라이브러리 불러오기

파이썬에서는 남이 만들어둔 기능 묶음을 `import`로 가져와서 쓴다.
- `pandas` : 엑셀 시트 같은 표(테이블) 데이터를 다루는 라이브러리. 거의 모든 데이터 분석의 기본 도구.
- `as pd` : `pandas`라는 긴 이름 대신 `pd`라는 짧은 별명으로 부르겠다는 뜻 (관례적으로 다들 이렇게 씀).

In [1]:
import pandas as pd

# 표를 출력할 때 열이 잘리지 않고 다 보이도록 설정 (컬럼 개수가 많은 파일이 있어서 필요함)
pd.set_option('display.max_columns', None)

# 파일들이 들어있는 폴더 경로를 변수로 저장해두면, 아래에서 경로를 여러 번 안 써도 됨
DATA_DIR = "../data"

# 주의: 이 CSV 파일들은 '한글 윈도우 표준 인코딩(cp949/euc-kr)'으로 저장되어 있음.
# 인코딩이란 '글자를 컴퓨터가 이해하는 숫자로 바꾸는 규칙'인데, 규칙이 다르면 한글이 깨져 보임.
# 그래서 pd.read_csv()를 부를 때마다 encoding="cp949"를 꼭 지정해줘야 함.
ENCODING = "cp949"

## 1. 상권 위치 정보 (master)

이 파일은 서울시 상권 하나하나가 **어디에 있는지**를 알려주는 '주소록' 같은 파일.
다른 두 파일(점포, 집객시설)에 나오는 `상권_코드`가 정확히 어느 동네인지 알고 싶을 때 이 파일을 찾아본다.

In [2]:
# pd.read_csv(파일경로, encoding=인코딩) : CSV 파일을 읽어서 DataFrame(표 형태 객체)으로 만든다.
# f"..." 형태는 f-string이라고 부르며, 문자열 안에 변수를 {} 로 끼워넣을 수 있게 해준다.
df_master = pd.read_csv(f"{DATA_DIR}/seoul_commercial_district_master_2025.csv", encoding=ENCODING)

# .shape : (행 개수, 열 개수)를 튜플로 알려줌. 표가 얼마나 큰지 가장 먼저 확인하는 습관을 들이자.
print("행(row) x 열(column) 개수:", df_master.shape)

행(row) x 열(column) 개수: (1650, 11)


In [3]:
# .head(5) : 위에서부터 5개 행만 미리보기. 전체를 다 보면 너무 많으니 일부만 확인.
df_master.head(5)

,상권_구분_코드,상권_구분_코드_명,상권_코드,상권_코드_명,엑스좌표_값,와이좌표_값,자치구_코드,자치구_코드_명,행정동_코드,행정동_코드_명,영역_면적
0,A,골목상권,3110055,황학동벼룩시장,201642,452260,11140,중구,11140670,황학동,27575
1,A,골목상권,3110008,배화여자대학교(박노수미술관),197093,453418,11110,종로구,11110515,청운효자동,149264
2,A,골목상권,3110009,자하문터널,196991,455057,11110,종로구,11110550,부암동,178306
3,A,골목상권,3110010,평창동서측,197064,456643,11110,종로구,11110560,평창동,369415
4,A,골목상권,3110017,정독도서관,198581,453781,11110,종로구,11110600,가회동,83855


In [4]:
# .info() : 각 열(column)의 이름, 데이터 개수(결측치 없는지), 데이터 타입(숫자인지 문자인지)을 한 번에 보여줌
df_master.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1650 entries, 0 to 1649
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   상권_구분_코드    1650 non-null   object
 1   상권_구분_코드_명  1650 non-null   object
 2   상권_코드       1650 non-null   int64 
 3   상권_코드_명     1650 non-null   object
 4   엑스좌표_값      1650 non-null   int64 
 5   와이좌표_값      1650 non-null   int64 
 6   자치구_코드      1650 non-null   int64 
 7   자치구_코드_명    1650 non-null   object
 8   행정동_코드      1650 non-null   int64 
 9   행정동_코드_명    1650 non-null   object
 10  영역_면적       1650 non-null   int64 
dtypes: int64(6), object(5)
memory usage: 141.9+ KB


### 컬럼 설명 (master)

| 컬럼명 | 의미 |
|---|---|
| 상권_구분_코드 | 상권의 종류를 나타내는 알파벳 코드 (A/D/R/U, 아래 표 참고) |
| 상권_구분_코드_명 | 위 코드의 한글 이름 |
| 상권_코드 | 상권 고유 번호. **다른 두 파일과 연결할 때 쓰는 열쇠(key)** |
| 상권_코드_명 | 상권 이름 (예: '황학동벼룩시장') |
| 엑스좌표_값 / 와이좌표_값 | 지도 상의 좌표 (위경도가 아니라 TM 좌표계로 추정 — 지도 시각화 시 변환 필요) |
| 자치구_코드 / 자치구_코드_명 | 서울시 25개 구 중 어디인지 (강남구, 종로구 등) |
| 행정동_코드 / 행정동_코드_명 | 더 세부 단위인 '동' |
| 영역_면적 | 상권 영역의 면적 |

**상권_구분_코드 4가지 종류**
- `A` = 골목상권 (동네 골목 단위의 작은 상권)
- `D` = 발달상권 (유동인구 많은 번화가급 상권)
- `R` = 전통시장
- `U` = 관광특구

In [5]:
# .value_counts() : 어떤 열에 어떤 값이 각각 몇 번씩 나오는지 개수를 세어줌
# → 상권 구분별로(골목/발달/전통시장/관광특구) 상권이 몇 개씩 있는지 확인
df_master["상권_구분_코드_명"].value_counts()

상권_구분_코드_명
골목상권    1090
전통시장     305
발달상권     249
관광특구       6
Name: count, dtype: int64

In [6]:
# 자치구(구)별로 상권이 몇 개씩 있는지 확인.
# .value_counts()는 기본적으로 개수가 많은 순서(내림차순)로 정렬해서 보여줌.
df_master["자치구_코드_명"].value_counts()

자치구_코드_명
강남구     103
영등포구     96
동대문구     77
마포구      77
성북구      76
서초구      74
관악구      74
종로구      74
송파구      70
강서구      70
중구       68
은평구      68
강북구      63
서대문구     62
광진구      62
중랑구      61
강동구      60
구로구      60
동작구      57
용산구      57
양천구      56
성동구      55
금천구      45
도봉구      43
노원구      42
Name: count, dtype: int64

## 2. 점포/업종 정보 (stores) — 제일 큰 파일 (약 30만 행)

상권마다, 분기마다, 업종마다 **점포가 몇 개 있었고 몇 개가 새로 생기고(개업) 몇 개가 문을 닫았는지(폐업)**를 담은 파일.
이 공모전의 핵심 데이터가 될 가능성이 높음 — '상권 쇠퇴'를 진단하려면 개업/폐업 흐름을 봐야 하니까.

In [7]:
df_stores = pd.read_csv(f"{DATA_DIR}/seoul_commercial_district_stores_2025.csv", encoding=ENCODING)
print("행 x 열 개수:", df_stores.shape)

행 x 열 개수: (304775, 12)


In [8]:
df_stores.head(5)

,기준_년분기_코드,상권_구분_코드,상권_구분_코드_명,상권_코드,상권_코드_명,서비스_업종_코드,서비스_업종_코드_명,전체_점포_수,일반_점포_수,프랜차이즈_점포_수,개업_점포_수,폐업_점포_수
0,20251,A,골목상권,3110001,이북5도청사,CS100001,한식음식점,12,13,0,8,1
1,20251,A,골목상권,3110001,이북5도청사,CS100003,일식음식점,1,1,0,0,0
2,20251,A,골목상권,3110001,이북5도청사,CS100008,분식전문점,2,2,0,0,0
3,20251,A,골목상권,3110001,이북5도청사,CS100009,호프-간이주점,2,3,0,33,1
4,20251,A,골목상권,3110001,이북5도청사,CS100010,커피-음료,2,2,0,0,0


In [9]:
df_stores.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 304775 entries, 0 to 304774
Data columns (total 12 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   기준_년분기_코드    304775 non-null  int64 
 1   상권_구분_코드     304775 non-null  object
 2   상권_구분_코드_명   304775 non-null  object
 3   상권_코드        304775 non-null  int64 
 4   상권_코드_명      304775 non-null  object
 5   서비스_업종_코드    304775 non-null  object
 6   서비스_업종_코드_명  304775 non-null  object
 7   전체_점포_수      304775 non-null  int64 
 8   일반_점포_수      304775 non-null  int64 
 9   프랜차이즈_점포_수   304775 non-null  int64 
 10  개업_점포_수      304775 non-null  int64 
 11  폐업_점포_수      304775 non-null  int64 
dtypes: int64(7), object(5)
memory usage: 27.9+ MB


### 컬럼 설명 (stores)

| 컬럼명 | 의미 |
|---|---|
| 기준_년분기_코드 | 예: 20251 = 2025년 1분기. 이 파일은 2025년 1~4분기 데이터를 담고 있음 |
| 상권_구분_코드 / 상권_구분_코드_명 | master 파일과 동일한 A/D/R/U 구분 |
| 상권_코드 / 상권_코드_명 | master 파일과 연결되는 열쇠 |
| 서비스_업종_코드 / 서비스_업종_코드_명 | 어떤 업종인지 (한식음식점, 미용실 등 총 100종) |
| 전체_점포_수 | 그 상권·그 분기·그 업종의 전체 점포 수 (= 일반 + 프랜차이즈) |
| 일반_점포_수 | 프랜차이즈가 아닌 개인/일반 점포 수 |
| 프랜차이즈_점포_수 | 프랜차이즈 가맹점 수 |
| 개업_점포_수 | 그 분기에 새로 생긴 점포 수 |
| 폐업_점포_수 | 그 분기에 문을 닫은 점포 수 |

In [10]:
# 어떤 분기 데이터가 들어있는지 확인 (2025년 1~4분기가 맞는지)
df_stores["기준_년분기_코드"].value_counts().sort_index()

기준_년분기_코드
20251    76383
20252    76238
20253    76169
20254    75985
Name: count, dtype: int64

In [11]:
# .nunique() : 그 열에 '서로 다른 값'이 몇 개나 있는지 세어줌 (중복 제외)
# → 서비스 업종이 총 몇 종류인지 확인
print("서비스 업종 종류 수:", df_stores["서비스_업종_코드_명"].nunique())

# .unique() : 실제 값 목록을 배열로 보여줌
df_stores["서비스_업종_코드_명"].unique()

서비스 업종 종류 수: 100


array(['한식음식점', '일식음식점', '분식전문점', '호프-간이주점', '커피-음료', '일반교습학원', '스포츠 강습',
       '한의원', '기타법무서비스', '당구장', '복권방', '미용실', '부동산중개업', '여관', '노래방',
       '독서실', '건축물청소', '편의점', '육류판매', '시계및귀금속', '의약품', '화장품', '조명용품',
       '예술품', '전자상거래업', '제과점', '패스트푸드점', '치킨전문점', '외국어학원', '예술학원', '치과의원',
       'PC방', '스포츠클럽', '자동차수리', '피부관리실', '세탁소', '가전제품수리', '여행사', '가정용품임대',
       '슈퍼마켓', '청과상', '반찬가게', '일반의류', '의료기기', '서적', '애완동물', '철물점', '악기',
       '인테리어', '일반의원', '변호사사무소', '골프연습장', '네일숍', '게스트하우스', '사진관',
       '통번역서비스', '주류도매', '미곡판매', '수산물판매', '문구', '섬유제품', '화초', '가전제품',
       '양식음식점', '세무사사무소', '미용재료', '모터사이클및부품', '자동차부품', '주유소', '자동차미용',
       '고시원', '녹음실', '핸드폰', '신발', '가방', '안경', '완구', '재생용품 판매점', '중식음식점',
       '유아의류', '자전거 및 기타운송장비', '중고가구', '운동/경기용품', '기타오락장', '모터사이클수리',
       '컴퓨터및주변장치판매', '의류임대', '한복점', '가구', '중고차판매', '통신기기수리', '비디오/서적임대',
       '컴퓨터학원', '동물병원', '법무사사무소', '회계사사무소', '변리사사무소', '전자게임장', '볼링장',
       'DVD방'], dtype=object)

In [12]:
# .describe() : 숫자로 된 열들의 통계 요약(평균, 최소/최대, 분포 등)을 한 번에 보여줌
# 전체_점포_수, 개업_점포_수, 폐업_점포_수가 보통 어느 정도 범위인지 감을 잡을 수 있음
df_stores[["전체_점포_수", "일반_점포_수", "프랜차이즈_점포_수", "개업_점포_수", "폐업_점포_수"]].describe()

,전체_점포_수,일반_점포_수,프랜차이즈_점포_수,개업_점포_수,폐업_점포_수
count,304775.000000,304775.000000,304775.000000,304775.000000,304775.000000
mean,6.257065,6.790308,1.987030,2.334539,0.533243
std,51.466804,51.863075,10.086123,11.706172,2.389032
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,1.000000,0.000000,0.000000,0.000000
50%,2.000000,2.000000,0.000000,0.000000,0.000000
75%,5.000000,5.000000,0.000000,0.000000,0.000000
max,11433.000000,11444.000000,200.000000,450.000000,151.000000


## 3. 집객시설 정보 (attraction_facilities)

상권 주변에 **사람을 끌어모으는 시설(지하철역, 학교, 병원, 백화점 등)**이 얼마나 있는지를 담은 파일.

⚠️ 주의: 이 데이터는 1년에 한 번(4분기)만 갱신되기 때문에, 같은 해의 1·2·3분기 값은 전부 동일함. 시계열(시간 흐름) 분석에는 쓸 수 없고, '이 상권 주변 인프라가 얼마나 풍부한가'라는 고정된 특성으로만 활용 가능.

In [13]:
# 이 파일만 유독 인코딩이 다르다 (utf-8-sig = UTF-8이면서 맨 앞에 BOM이라는 표시가 붙은 형식).
# 같은 폴더 안에서도 파일마다 저장 방식이 다를 수 있다는 걸 보여주는 사례.
df_facilities = pd.read_csv(
    f"{DATA_DIR}/seoul_commercial_district_attraction_facilities_2025.csv",
    encoding="utf-8-sig",
)
print("행 x 열 개수:", df_facilities.shape)

행 x 열 개수: (6312, 25)


In [14]:
df_facilities.head(5)

,기준_년분기_코드,상권_구분_코드,상권_구분_코드_명,상권_코드,상권_코드_명,집객시설_수,관공서_수,은행_수,종합병원_수,일반_병원_수,약국_수,유치원_수,초등학교_수,중학교_수,고등학교_수,대학교_수,백화점_수,슈퍼마켓_수,극장_수,숙박_시설_수,공항_수,철도_역_수,버스_터미널_수,지하철_역_수,버스_정거장_수
0,20254,U,관광특구,3001496,강남 마이스 관광특구,35,1.0,8.0,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,8.0
1,20254,U,관광특구,3001495,잠실 관광특구,158,3.0,13.0,NaN,NaN,14.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,3.0,20.0
2,20254,U,관광특구,3001494,종로?청계 관광특구,246,5.0,17.0,NaN,NaN,32.0,NaN,NaN,NaN,NaN,NaN,NaN,5.0,2.0,15.0,NaN,NaN,NaN,3.0,28.0
3,20254,U,관광특구,3001493,동대문패션타운 관광특구,144,3.0,12.0,1.0,1.0,15.0,NaN,1.0,1.0,3.0,NaN,NaN,3.0,2.0,1.0,NaN,NaN,NaN,5.0,17.0
4,20254,U,관광특구,3001492,명동 남대문 북창동 다동 무교동 관광특구,594,14.0,57.0,NaN,NaN,43.0,NaN,NaN,NaN,NaN,1.0,1.0,2.0,5.0,9.0,NaN,NaN,NaN,3.0,44.0


In [15]:
df_facilities.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6312 entries, 0 to 6311
Data columns (total 25 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   기준_년분기_코드   6312 non-null   int64  
 1   상권_구분_코드    6312 non-null   object 
 2   상권_구분_코드_명  6312 non-null   object 
 3   상권_코드       6312 non-null   int64  
 4   상권_코드_명     6312 non-null   object 
 5   집객시설_수      6312 non-null   int64  
 6   관공서_수       2520 non-null   float64
 7   은행_수        1948 non-null   float64
 8   종합병원_수      92 non-null     float64
 9   일반_병원_수     528 non-null    float64
 10  약국_수        4036 non-null   float64
 11  유치원_수       604 non-null    float64
 12  초등학교_수      52 non-null     float64
 13  중학교_수       16 non-null     float64
 14  고등학교_수      44 non-null     float64
 15  대학교_수       336 non-null    float64
 16  백화점_수       44 non-null     float64
 17  슈퍼마켓_수      420 non-null    float64
 18  극장_수        336 non-null    float64
 19  숙박_시설_수     580 non-null   

### 컬럼 설명 (attraction_facilities)

| 컬럼명 | 의미 |
|---|---|
| 기준_년분기_코드, 상권_구분_코드(_명), 상권_코드(_명) | 위 두 파일과 동일한 개념 |
| 집객시설_수 | 아래 세부 시설들을 다 합친 총합 |
| 관공서_수, 은행_수, 종합병원_수, 일반_병원_수, 약국_수 | 행정/의료/금융 시설 |
| 유치원_수, 초등학교_수, 중학교_수, 고등학교_수, 대학교_수 | 교육 시설 |
| 백화점_수, 슈퍼마켓_수, 극장_수, 숙박_시설_수 | 상업/여가 시설 |
| 공항_수, 철도_역_수, 버스_터미널_수, 지하철_역_수, 버스_정거장_수 | 교통 시설 |

세부 시설 수 컬럼들은 해당 시설이 하나도 없는 상권에서는 빈 값(결측치)으로 표시되어 있을 수 있음 → 분석 시 0으로 채워야 함 (아직 이 노트북에서는 채우지 않고 원본 그대로만 확인).

In [16]:
# .isnull() : 각 칸이 결측치(비어있는 값)인지 아닌지를 True/False로 표시
# .sum() : True를 1로 취급해서 다 더함 → 열별로 결측치가 몇 개인지 셀 수 있음
df_facilities.isnull().sum()

기준_년분기_코드        0
상권_구분_코드         0
상권_구분_코드_명       0
상권_코드            0
상권_코드_명          0
집객시설_수           0
관공서_수         3792
은행_수          4364
종합병원_수        6220
일반_병원_수       5784
약국_수          2276
유치원_수         5708
초등학교_수        6260
중학교_수         6296
고등학교_수        6268
대학교_수         5976
백화점_수         6268
슈퍼마켓_수        5892
극장_수          5976
숙박_시설_수       5732
공항_수          6308
철도_역_수        6312
버스_터미널_수      6304
지하철_역_수       5532
버스_정거장_수      1232
dtype: int64

## 4. 세 파일을 연결(조인)해보는 맛보기

실제 분석에서는 세 파일을 합쳐서 봐야 의미가 생긴다.
예: "점포 데이터의 상권_코드가 실제로 어느 자치구에 있는 상권인지" 알고 싶으면 master 파일과 합쳐야 함.

여기서는 딱 한 번, 아주 간단하게 합치는 예시만 보여줌 (본격 분석은 나중에).

In [17]:
# pd.merge(왼쪽표, 오른쪽표, on="공통열", how="방식")
# - on="상권_코드" : 두 표에서 이 열의 값이 같은 행끼리 짝지어 합친다
# - how="left" : 왼쪽 표(df_stores) 기준으로 합치고, 오른쪽에 짝이 없으면 빈 값으로 둠
#
# df_master에서 필요한 열만 골라서 합친다: [[...]] 로 여러 열을 리스트로 선택
df_merged_sample = pd.merge(
    df_stores.head(1000),  # 30만 행 전체 대신, 맛보기로 앞 1000행만 사용 (컴퓨터 부담 줄이기)
    df_master[["상권_코드", "자치구_코드_명", "행정동_코드_명"]],
    on="상권_코드",
    how="left",
)

# 원래 stores에는 없던 '자치구_코드_명', '행정동_코드_명' 열이 새로 붙은 걸 확인
df_merged_sample.head(5)

,기준_년분기_코드,상권_구분_코드,상권_구분_코드_명,상권_코드,상권_코드_명,서비스_업종_코드,서비스_업종_코드_명,전체_점포_수,일반_점포_수,프랜차이즈_점포_수,개업_점포_수,폐업_점포_수,자치구_코드_명,행정동_코드_명
0,20251,A,골목상권,3110001,이북5도청사,CS100001,한식음식점,12,13,0,8,1,종로구,평창동
1,20251,A,골목상권,3110001,이북5도청사,CS100003,일식음식점,1,1,0,0,0,종로구,평창동
2,20251,A,골목상권,3110001,이북5도청사,CS100008,분식전문점,2,2,0,0,0,종로구,평창동
3,20251,A,골목상권,3110001,이북5도청사,CS100009,호프-간이주점,2,3,0,33,1,종로구,평창동
4,20251,A,골목상권,3110001,이북5도청사,CS100010,커피-음료,2,2,0,0,0,종로구,평창동


## 정리

- `df_master` : 1,650개 상권의 위치 정보 (자치구/행정동/좌표)
- `df_stores` : 약 30만 행, 상권×분기×업종별 점포 수/개폐업 정보 (제일 중요한 분석 대상)
- `df_facilities` : 상권별 주변 인프라(시설) 개수, 연 1회만 갱신됨
- 세 파일 모두 `상권_코드`로 연결 가능

다음 단계는 이 노트북 검토 후, 실제로 '상권 쇠퇴'를 어떻게 정의하고 진단할지 논의하는 것.